In [257]:
import sys
sys.path.append("/home/kp_info_loader")


from exchange_plugin.okx_plug import InitOkxAdaptor
from exchange_plugin.upbit_plug import InitUpbitAdaptor
from exchange_plugin.binance_plug import InitBinanceAdaptor
from exchange_plugin.bithumb_plug import InitBithumbAdaptor
from exchange_plugin.bybit_plug import InitBybitAdaptor
from etc.db_handler.mongodb_client import InitDBClient # TEMP
from etc.register_monitor_msg import RegisterMonitorMsg # TEMP
import pandas as pd
import json
import time
import datetime
from threading import Thread
import numpy as np

info_dict = {}
info_thread_dict = {}

okx_adaptor = InitOkxAdaptor("fcb66a6e-0443-4743-8d9b-61caf7eab662", "0029E09874B38F5AC7E68E9DFC348667", "121431Rn!!")
upbit_adaptor = InitUpbitAdaptor("x2AKLfsRtAgRxFjQ7p9TZTAg6E1SveoqfNfD8MK8", "svEran52QdsUyJdxAPYoTgFT2MXsHc5ZiKsKPxTL", info_dict, None)
binance_adaptor = InitBinanceAdaptor("4PFfYsObYyUaMlk7m6tT9qIl8X8P3WCUsyu1lEyZ4h50VfTMPIQCNdL9eOJCl3Lb", "011Z1W9p4425AZgtCNbs5L1d77ehZFNmBIwT0pz1SJGP7EiOlPILfWBglbVsxMcK", info_dict, None)
bithumb_adaptor = InitBithumbAdaptor()
bybit_adaptor = InitBybitAdaptor()

# UPBIT SPOT (KRW, BTC Market)
# UPBIT wallet status
# BINANCE SPOT, USD-M Futures, COIN-M Futures
data_name_list = [
        "upbit_spot_info_df",
        "upbit_spot_ticker_df",
        # "upbit_wallet_status_df",
        "binance_spot_ticker_df",
        "binance_spot_info_df",
        "binance_usd_m_ticker_df",
        "binance_usd_m_info_df",
        "binance_coin_m_ticker_df",
        "binance_coin_m_info_df",
        "okx_spot_ticker_df",
        "okx_spot_info_df",
        "okx_usd_m_ticker_df",
        "okx_usd_m_info_df",
        "okx_coin_m_ticker_df",
        "okx_coin_m_info_df",
        "bithumb_spot_info_df",
        "bithumb_spot_ticker_df",
        # "bithumb_wallet_status_df",
        "bybit_spot_info_df",
        "bybit_spot_ticker_df",
        "bybit_usd_m_info_df",
        "bybit_usd_m_ticker_df",
        "bybit_coin_m_info_df",
        "bybit_coin_m_ticker_df"
    ]

def update_exchange_info_as_df(data_name, loop_time_secs=15):
    if data_name == "upbit_spot_ticker_df":
        info_dict[data_name] = upbit_adaptor.spot_all_tickers()
    elif data_name == "upbit_wallet_status_df":
        info_dict[data_name] = upbit_adaptor.wallet_status()
    elif data_name == "upbit_spot_info_df":
        info_dict[data_name] = upbit_adaptor.spot_exchange_info()
    elif data_name == "binance_spot_ticker_df":
        info_dict[data_name] = binance_adaptor.spot_all_tickers()
    elif data_name == "binance_spot_info_df":
        info_dict[data_name] = binance_adaptor.spot_exchange_info()
    elif data_name == "binance_usd_m_ticker_df":
        info_dict[data_name] = binance_adaptor.usd_m_all_tickers()
    elif data_name == "binance_usd_m_info_df":
        info_dict[data_name] = binance_adaptor.usd_m_exchange_info()
    elif data_name == "binance_coin_m_ticker_df":
        info_dict[data_name] = binance_adaptor.coin_m_all_tickers()
    elif data_name == "binance_coin_m_info_df":
        info_dict[data_name] = binance_adaptor.coin_m_exchange_info()
    elif data_name == "okx_spot_ticker_df":
        info_dict[data_name] = okx_adaptor.spot_all_tickers()
    elif data_name == "okx_spot_info_df":
        info_dict[data_name] = okx_adaptor.spot_exchange_info()
    elif data_name == "okx_usd_m_ticker_df":
        info_dict[data_name] = okx_adaptor.usd_m_all_tickers()
    elif data_name == "okx_usd_m_info_df":
        info_dict[data_name] = okx_adaptor.usd_m_exchange_info()
    elif data_name == "okx_coin_m_ticker_df":
        info_dict[data_name] = okx_adaptor.coin_m_all_tickers()
    elif data_name == "okx_coin_m_info_df":
        info_dict[data_name] = okx_adaptor.coin_m_exchange_info()
    elif data_name == "bithumb_spot_info_df":
        info_dict[data_name] = bithumb_adaptor.spot_exchange_info()
    elif data_name == "bithumb_spot_ticker_df":
        info_dict[data_name] = bithumb_adaptor.spot_all_tickers()
    elif data_name == "bithumb_wallet_status_df":
        info_dict[data_name] = bithumb_adaptor.wallet_status()
    elif data_name == "bybit_spot_info_df":
        info_dict[data_name] = bybit_adaptor.spot_exchange_info()
    elif data_name == "bybit_spot_ticker_df":
        info_dict[data_name] = bybit_adaptor.spot_all_tickers()
    elif data_name == "bybit_usd_m_info_df":
        info_dict[data_name] = bybit_adaptor.usd_m_exchange_info()
    elif data_name == "bybit_usd_m_ticker_df":
        info_dict[data_name] = bybit_adaptor.usd_m_all_tickers()
    elif data_name == "bybit_coin_m_info_df":
        info_dict[data_name] = bybit_adaptor.coin_m_exchange_info()
    elif data_name == "bybit_coin_m_ticker_df":
        info_dict[data_name] = bybit_adaptor.coin_m_all_tickers()
    else:
        print(f"update_exchange_info_as_df|name:{data_name} is not valid.")

for data_name in data_name_list:
    info_thread_dict[f"update_{data_name}"] = Thread(target=update_exchange_info_as_df, args=(data_name,), daemon=True)
    info_thread_dict[f"update_{data_name}"].start()
    print(f"InitCore|update_{data_name} thread has started.")


with open("/home/kp_info_loader/kp_info_loader_config.json") as f:
        config = json.load(f)

node = config['node']
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id_list = []
admin_id = config['telegram_admin_id']['charlie1155']
admin_id_list.append(admin_id)
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir=None)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

enabled_market_klines = config['node_settings'][node]['enabled_market_klines']
# total enabled market settings
total_enabled_market_klines = []
for each_market_code_list in config['node_settings'].values():
    total_enabled_market_klines += each_market_code_list['enabled_market_klines']

2023-11-22 18:51:05,820 - okx_plug - INFO - okx_plug_logger started.
2023-11-22 18:51:05,820 - okx_plug - INFO - okx_plug_logger started.
2023-11-22 18:51:06,518 - upbit_plug - INFO - upbit_plug_logger started.
2023-11-22 18:51:06,518 - upbit_plug - INFO - upbit_plug_logger started.
2023-11-22 18:51:06,661 - binance_plug - INFO - binance_plug_logger started.
2023-11-22 18:51:06,661 - binance_plug - INFO - binance_plug_logger started.
2023-11-22 18:51:07,067 - bithumb_plug - INFO - bithumb_plug_logger started.
2023-11-22 18:51:07,067 - bithumb_plug - INFO - bithumb_plug_logger started.
2023-11-22 18:51:07,071 - bybit_plug - INFO - bybit_plug_logger started.
2023-11-22 18:51:07,071 - bybit_plug - INFO - bybit_plug_logger started.


InitCore|update_upbit_spot_info_df thread has started.
InitCore|update_upbit_spot_ticker_df thread has started.
InitCore|update_binance_spot_ticker_df thread has started.
InitCore|update_binance_spot_info_df thread has started.
InitCore|update_binance_usd_m_ticker_df thread has started.
InitCore|update_binance_usd_m_info_df thread has started.
InitCore|update_binance_coin_m_ticker_df thread has started.
InitCore|update_binance_coin_m_info_df thread has started.
InitCore|update_okx_spot_ticker_df thread has started.
InitCore|update_okx_spot_info_df thread has started.
InitCore|update_okx_usd_m_ticker_df thread has started.
InitCore|update_okx_usd_m_info_df thread has started.
InitCore|update_okx_coin_m_ticker_df thread has started.
InitCore|update_okx_coin_m_info_df thread has started.
InitCore|update_bithumb_spot_info_df thread has started.
InitCore|update_bithumb_spot_ticker_df thread has started.
InitCore|update_bybit_spot_info_df thread has started.
InitCore|update_bybit_spot_ticker

2023-11-22 18:51:07,849 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_coin_m_info_df') is None, Fetched from API
2023-11-22 18:51:07,849 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_coin_m_info_df') is None, Fetched from API
2023-11-22 18:51:11,059 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_spot_info_df') is None, Fetched from API
2023-11-22 18:51:11,059 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_spot_info_df') is None, Fetched from API
2023-11-22 18:51:11,251 - binance_plug - INFO - self.info_dict is None or self.info_dict.get('binance_usd_m_info_df') is None, Fetched from API
2023-11-22 18:51:11,251 - binance_plug - INFO - self.info_dict is None or self.info_dict.get('binance_usd_m_info_df') is None, Fetched from API
2023-11-22 18:51:11,438 - bybit_plug - INFO - self.info_dict is None or self.info_dict.get('bybit_usd_m_info_df') is None, Fetched from API
2023-11-22 1

In [259]:
exchange_list = []
for each_market_code_combi in total_enabled_market_klines:
    first_market_code, second_market_code = each_market_code_combi.split(":")
    first_market, first_quote_asset = first_market_code.split('/')
    first_exchange = first_market.split('_')[0]
    first_market_type = first_market.replace(f'{first_exchange}_','')
    second_market, second_quote_asset = second_market_code.split('/')
    second_exchange = second_market_code.split('_')[0]
    second_market_type = second_market.replace(f'{second_exchange}_','')
    if first_market_type in ["USD_M", "COIN_M"]:
        exchange_list.append(first_exchange)
    if second_market_type in ["USD_M", "COIN_M"]:
        exchange_list.append(second_exchange)
exchange_list = list(set(exchange_list))
perpetual_tup_list = []
for each_exchange in exchange_list:
    for each_market_type in ['USD_M', 'COIN_M']:
        perpetual_tup_list.append((each_exchange, each_market_type))

perpetual_combination_list = []
for i, perpetual_tup_one in enumerate(perpetual_tup_list):
    for perpetual_tup_two in perpetual_tup_list[i+1:]:
        perpetual_combination_list.append((perpetual_tup_one, perpetual_tup_two))

perpetual_combination_list

[(('BINANCE', 'USD_M'), ('BINANCE', 'COIN_M')),
 (('BINANCE', 'USD_M'), ('OKX', 'USD_M')),
 (('BINANCE', 'USD_M'), ('OKX', 'COIN_M')),
 (('BINANCE', 'USD_M'), ('BYBIT', 'USD_M')),
 (('BINANCE', 'USD_M'), ('BYBIT', 'COIN_M')),
 (('BINANCE', 'COIN_M'), ('OKX', 'USD_M')),
 (('BINANCE', 'COIN_M'), ('OKX', 'COIN_M')),
 (('BINANCE', 'COIN_M'), ('BYBIT', 'USD_M')),
 (('BINANCE', 'COIN_M'), ('BYBIT', 'COIN_M')),
 (('OKX', 'USD_M'), ('OKX', 'COIN_M')),
 (('OKX', 'USD_M'), ('BYBIT', 'USD_M')),
 (('OKX', 'USD_M'), ('BYBIT', 'COIN_M')),
 (('OKX', 'COIN_M'), ('BYBIT', 'USD_M')),
 (('OKX', 'COIN_M'), ('BYBIT', 'COIN_M')),
 (('BYBIT', 'USD_M'), ('BYBIT', 'COIN_M'))]